In [0]:
## trips reading and saving as delta tables
raw_trips_df = spark.read \
    .format("parquet") \
    .load("/Volumes/nyc/default/nyc_uber/*.parquet")

raw_trips_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc.default.bronze_fhv_trips")

In [0]:
##zone tables reading and saving as delta tables
raw_zones_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/nyc/default/nyc_uber/taxi_zone_lookup.csv")

raw_zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc.default.bronze_taxi_zones")

In [0]:
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, lit, current_timestamp

bronze_trips = spark.read.table("nyc.default.bronze_fhv_trips")
bronze_zones = spark.read.table("nyc.default.bronze_taxi_zones")
total_bronze_rows = bronze_trips.count()
display(total_bronze_rows)

In [0]:
##joining zone lookups

pu_zones = bronze_zones.select(
    col("LocationID").alias("PU_LocationID"),
    col("Borough").alias("PU_borough"),
    col("Zone").alias("PU_zone"),
    col("service_zone").alias("PU_service_zone")
)

do_zones = bronze_zones.select(
    col("LocationID").alias("DO_LocationID"),
    col("Borough").alias("DO_borough"),
    col("Zone").alias("DO_zone"),
    col("service_zone").alias("DO_service_zone")
)

joined_df = bronze_trips.join(
    pu_zones, bronze_trips["PUlocationID"] == pu_zones["PU_LocationID"], "left"
).drop("PU_LocationID")

joined_df = joined_df.join(
    do_zones, joined_df["DOlocationID"] == do_zones["DO_LocationID"], "left"
).drop("DO_LocationID")

In [0]:
## standrize date and time 
timestamp_df = joined_df \
    .withColumn("pickup_datetime", to_timestamp(col("pickup_datetime"))) \
    .withColumn("dropOff_datetime", to_timestamp(col("dropOff_datetime")))

##removing the duplicates

deduped_df = timestamp_df.dropDuplicates()
deduped_rows_count = deduped_df.count()
duplicate_rows_dropped = total_bronze_rows - deduped_rows_count

In [0]:
##remove unlogical rows (dropOff <= pickup)
cleaned_df = deduped_df.filter(
    (col("pickup_datetime").isNotNull()) & 
    (col("dropOff_datetime").isNotNull()) & 
    (col("dropOff_datetime") > col("pickup_datetime"))
)
valid_rows_count = cleaned_df.count()
invalid_time_rows_dropped = deduped_rows_count - valid_rows_count

## fill in the null values
silver_final = cleaned_df.fillna({
    "PUlocationID": 300,
    "DOlocationID": 300,
    "SR_Flag": 0,
    "PU_borough": "Unknown",
    "PU_zone": "Unknown",
    "PU_service_zone": "Unknown",
    "DO_borough": "Unknown",
    "DO_zone": "Unknown",
    "DO_service_zone": "Unknown"
})

##save the table
silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc.default.silver_fhv_trips")